In [ ]:
import sys
if "google.colab" in sys.modules:
    !pip install -q -U unsloth trl datasets transformers

### Listing 6.25: Library Imports and Hardware Precision Setup

In [ ]:
import os
import torch
from datasets import load_dataset

from trl import SFTTrainer, SFTConfig

compute_dtype = (
    torch.bfloat16
    if torch.backends.mps.is_available()
    else (torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8 else torch.float16)
)


### Listing 6.26: Loading the Vision-Language Model and Configuring LoRA Adapters

In [ ]:
# macOS Apple Silicon Adaptation:
# Unsloth is CUDA-only; on macOS we load with Hugging Face Transformers + PEFT.
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from peft import LoraConfig, get_peft_model

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
device = "mps" if torch.backends.mps.is_available() else "cpu"

print(f"Loading {MODEL_ID} on {device}...")
processor = AutoProcessor.from_pretrained(MODEL_ID)
tokenizer = processor.tokenizer
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    dtype=compute_dtype,
    device_map=device,
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()


### Listing 6.28: Configuring SFTTrainer and Fine-Tuning Qwen2-VL

In [ ]:
dataset = load_dataset("unsloth/LaTeX_OCR", split="train")

shuffled_dataset = dataset.shuffle(seed=42)
train_ds = shuffled_dataset.select(range(500))
eval_ds = shuffled_dataset.select(range(500, 550))


def convert_to_conversation(sample):
    conversation = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Write the LaTeX representation for this image.",
                },
                {"type": "image"},
            ],
        },
        {"role": "assistant", "content": [{"type": "text", "text": sample["text"]}]},
    ]
    return {"messages": conversation, "images": [sample["image"]]}


train_mapped = train_ds.map(
    convert_to_conversation, remove_columns=train_ds.column_names
)
eval_mapped = eval_ds.map(convert_to_conversation, remove_columns=eval_ds.column_names)

### Listing 6.28: Configuring SFTTrainer and Fine-Tuning Qwen2-VL

In [ ]:
def collate_fn(batch):
    texts = [
        processor.apply_chat_template(
            example["messages"], tokenize=False, add_generation_prompt=False
        )
        for example in batch
    ]
    image_inputs = [example["images"][0] for example in batch]
    batch_encodings = processor(
        text=texts, images=image_inputs, return_tensors="pt", padding=True
    )
    labels = batch_encodings["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    batch_encodings["labels"] = labels
    return batch_encodings

training_args = SFTConfig(
    output_dir="qwen2-vl-latex",
    dataset_text_field="",
    dataset_kwargs={"skip_prepare_dataset": True},
    max_length=512,
    remove_unused_columns=False,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=0,
    max_steps=30,
    optim="adamw_torch",
    bf16=(compute_dtype == torch.bfloat16),
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=50,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    data_collator=collate_fn,
    train_dataset=train_mapped,
    eval_dataset=eval_mapped,
    args=training_args,
)

model.config.use_cache = False

trainer.train()

model.save_pretrained("qwen2-vl-latex-adapter")
processor.save_pretrained("qwen2-vl-latex-adapter")


### Listing 6.29: Running Inference and Evaluating the Fine-Tuned Vision Model

In [ ]:
model.eval()
model.generation_config.max_length = None

sample = eval_ds[0]
image = sample["image"]
expected_latex = sample["text"]

messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Write the LaTeX representation for this image."},
            {"type": "image"},
        ],
    }
]

input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=[input_text], images=[image], return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=128)

generated_text = processor.decode(
    outputs[0][inputs.input_ids.shape[1] :], skip_special_tokens=True
).strip()

print("--- Expected LaTeX ---")
print(expected_latex)
print("--- Generated LaTeX ---")
print(generated_text)
